# Занятие 4. Машинное обучение на данных розничной сети

Три задачи — три типа моделей и три места, где их обучают:

| часть | тип задачи | вопрос бизнеса | объект | где обучаем |
|---|---|---|---|---|
| 1 | **регрессия** | какую выручку магазин сделает за час? | магазин × час | **sklearn** в ноутбуке |
| 2 | **классификация** | похож ли покупатель на участника бонусной программы? | чек | **Spark ML** в ноутбуке |
| 3 | **кластеризация** | на какие группы делятся магазины сети? | магазин | **Spark ML в Airflow**, раз в неделю |
| 4 | — | показать сегменты магазинов руководству | — | дашборд в **Metabase** |

Путь данных:

```
iceberg.retail.sales_silver ──Spark SQL──▶ датасет (одна строка = один объект) ──▶ модель ──▶ прогнозы / сегменты
```

**Перед началом:** таблица `iceberg.retail.sales_silver` заполнена (занятие 2).

In [ ]:
import sys
sys.path.append("../host")
sys.path.append("../jobs/retail")   # скрипт DAG'а кластеризации: импортируем его функции в части 3
from spark_session import get_spark
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt

spark = get_spark("lab-04")

DWH = dict(url="jdbc:postgresql://postgres:5433/dwh", user="course", password="course_pass",
           driver="org.postgresql.Driver")


def read_from_dwh(table):
    return spark.read.format("jdbc").options(dbtable=f"public.{table}", **DWH).load()

## Часть 1. Регрессия в sklearn: выручка магазина за час

Директору магазина нужно планировать смены кассиров. Модель по **формату** магазина,
**региону** и **часу** прогнозирует выручку магазина за этот час.

* **Объект** — пара «магазин × час».
* **Целевая переменная** (что прогнозируем) — `revenue_rub`, число → это **регрессия**.
* **Признаки** (что известно заранее) — `format`, `region`, `hour_msk`.

sklearn работает в памяти одного компьютера. Поэтому сначала Spark сворачивает 35 млн
строк silver в маленький датасет (несколько тысяч строк), и только он попадает в pandas
через `toPandas()`.

### Задача 1.1. Соберите датасет

Одна строка — один магазин за один час одного дня:

| колонка | смысл |
|---|---|
| `store_id` | магазин |
| `format`, `region` | формат и регион магазина из `iceberg.retail.stores` |
| `event_date` | дата |
| `hour_msk` | час по Москве: `hour(event_ts)` |
| `revenue_rub` | выручка продаж за час, руб.: сумма `price_paid_kop` неотменённых продаж / 100, тип `DOUBLE` |

*Подсказки.* Только `SALE` и `NOT is_cancelled`, возвраты в этой задаче не учитываем.
`GROUP BY` по всем колонкам, кроме выручки. `DOUBLE`, а не `DECIMAL`: `DECIMAL` в pandas
превращается в объекты `Decimal`, с которыми sklearn работает медленно.

Ожидается около 6 900 строк: в часы, когда магазин закрыт, строк нет.

In [ ]:
# Задача 1.1
store_hours = spark.sql("""
    -- ваш запрос
""").toPandas()

print(len(store_hours), "строк")
store_hours.head()

Прежде чем учить модель — посмотрите на данные. Средняя выручка магазина по часам:

In [ ]:
(store_hours.groupby(["hour_msk", "format"])["revenue_rub"].mean().unstack().div(1000)
    .plot(figsize=(10, 4), marker="o", xlabel="час, МСК", ylabel="тыс. руб.",
          title="Средняя выручка магазина за час"))
plt.show()

Вопросы: в каком формате выручка за час больше всего? Откуда выручка в 3–5 часов ночи
по Москве?

### Признаки и разбиение на train и test

Модели sklearn понимают только числа. Текстовые колонки превращаем в набор колонок 0/1
(**one-hot**): `format_супермаркет = 1`, если магазин — супермаркет. `drop_first=True`
выбрасывает первую категорию каждого признака — она становится точкой отсчёта
(гипермаркет и первый по алфавиту регион).

Качество проверяем на данных, которых модель не видела. Разбиваем **по времени**: учимся
на 4–5 сентября, проверяем на 6-м — так модель будет работать и в жизни: учится на
прошлом, прогнозирует будущее.

In [ ]:
X = pd.get_dummies(store_hours[["format", "region"]], drop_first=True, dtype=int)
X["hour_msk"] = store_hours["hour_msk"]
y = store_hours["revenue_rub"]

is_train = store_hours["event_date"].astype(str) < "2026-09-06"
X_train, y_train = X[is_train], y[is_train]
X_test, y_test = X[~is_train], y[~is_train]

print("train:", len(X_train), " test:", len(X_test), " признаков:", X.shape[1])
X.head()

### Пример: базовая линия и линейная регрессия

**Базовая линия** — самый простой прогноз: средняя выручка по train для любого часа
любого магазина. Модель имеет смысл, только если она заметно лучше.

Метрики:

* **MAE** — средняя ошибка в рублях: насколько в среднем прогноз отличается от факта.
* **R²** — какую долю разброса выручки объясняет модель: 1 — идеально, 0 — не лучше среднего.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score


def report(name, y_true, y_pred):
    print(f"{name:22s} MAE = {mean_absolute_error(y_true, y_pred):>10,.0f} руб.   "
          f"R² = {r2_score(y_true, y_pred):.3f}")


report("среднее (базовая линия)", y_test, [y_train.mean()] * len(y_test))

linear = LinearRegression().fit(X_train, y_train)
report("линейная регрессия", y_test, linear.predict(X_test))

Линейная модель интерпретируется напрямую: коэффициент — на сколько рублей меняется
прогноз, если признак равен 1 (или вырос на единицу), при прочих равных.

In [ ]:
pd.Series(linear.coef_, index=X.columns).sort_values().round(-3).to_frame("руб.")

Вопросы: насколько супермаркет и магазин «у дома» отстают от гипермаркета? Что модель
думает про `hour_msk` — и почему один коэффициент не может описать форму кривой
с графика выше?

### Задача 1.2. Случайный лес

Случайный лес — много решающих деревьев, прогноз — их среднее. Дерево задаёт вопросы
вида «час ≥ 17?», «формат = гипермаркет?», поэтому само находит нелинейности (вечерний пик)
и сочетания признаков (ночь по Москве + Хабаровск).

Обучите `RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1)`
на `X_train`, `y_train` и сравните с линейной регрессией.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Задача 1.2
forest = ...   # создайте модель и обучите её методом fit

report("случайный лес", y_test, forest.predict(X_test))

**Важность признаков** показывает, насколько каждый признак уменьшает ошибку деревьев
(сумма по всем признакам — 1):

In [ ]:
pd.Series(forest.feature_importances_, index=X.columns).sort_values(ascending=False).head(10).round(3)

Прогноз и факт для одного магазина в тестовый день:

In [ ]:
store_id = 1
mask = (~is_train) & (store_hours["store_id"] == store_id)
(store_hours[mask].assign(forecast=forest.predict(X[mask])).sort_values("hour_msk")
    .plot(x="hour_msk", y=["revenue_rub", "forecast"], marker="o", figsize=(10, 4),
          title=f"Магазин {store_id}, 6 сентября: факт и прогноз"))
plt.show()

Вопросы:

1. Во сколько раз ошибка леса меньше ошибки линейной модели? Почему?
2. Какой признак важнее всего?
3. Поменяйте `store_id`. Для некоторых магазинов прогноз систематически выше или ниже
   факта. Почему? Каких сведений о магазине модели не хватает?

## Часть 2. Классификация в Spark ML: участник программы лояльности?

Без карты программы проходит почти половина чеков. Если модель найдёт среди них чеки,
**похожие** на чеки участников, кассир сможет предложить карту именно этим покупателям.

* **Объект** — чек (продажи 5 сентября: около 1,8 млн чеков).
* **Целевая переменная** — `label`: 1, если покупатель участник программы, иначе 0 →
  **бинарная классификация**.
* **Признаки** — что видно в самом чеке: число позиций, сумма, скидка, способ оплаты,
  формат магазина, час.

Почти 2 млн строк в pandas уже неудобно, поэтому датасет остаётся в Spark, и модель
обучается на кластере библиотекой **Spark ML**.

**Утечка данных.** `customer_id` в признаки не берём: участник всегда показывает карту,
поэтому «покупатель известен» почти совпадает с ответом. Модель с таким признаком
показала бы отличное качество на истории и была бы бесполезна в деле: у анонимных чеков,
к которым мы хотим её применять, `customer_id` всегда пуст.

### Задача 2.1. Соберите датасет чеков

| колонка | смысл |
|---|---|
| `receipt_id` | чек |
| `label` | `MAX(CAST(is_loyalty AS INT))` — 1 для участника программы |
| `items` | число позиций |
| `amount_rub` | сумма чека, руб., `DOUBLE` |
| `discount_pct` | скидка по чеку, %: `100 * (1 - SUM(price_paid_kop) / SUM(price_regular_kop))` |
| `payment_method` | способ оплаты: `FIRST(payment_method)` — он общий для всего чека |
| `format` | формат магазина |
| `hour_msk` | час начала чека: `hour(MIN(event_ts))` |

*Подсказки.* Только неотменённые продажи (`SALE`, `NOT is_cancelled`) за
`event_date = DATE'2026-09-05'`, `GROUP BY` по чеку и формату.

In [ ]:
# Задача 2.1
receipts = spark.sql("""
    -- ваш запрос
""").cache()

print(receipts.count(), "чеков")
receipts.show(5)

Чем отличаются чеки участников? Если средние признаков в двух классах одинаковые,
модели не за что зацепиться.

In [ ]:
(receipts.groupBy("label")
    .agg(F.count("*").alias("receipts"), F.avg("items").alias("items"),
         F.avg("amount_rub").alias("amount_rub"), F.avg("discount_pct").alias("discount_pct"))
    .toPandas().round(2))

In [ ]:
receipts.groupBy("payment_method").pivot("label").count().show()

### Pipeline в Spark ML

Модель Spark ML — цепочка этапов (**Pipeline**), каждый добавляет к DataFrame колонку:

| этап | что делает |
|---|---|
| `StringIndexer` | строку в номер: `CARD` → 0, `SBP` → 1, … |
| `OneHotEncoder` | номер в вектор 0/1, как `get_dummies` в части 1 (последняя категория — точка отсчёта) |
| `VectorAssembler` | все признаки в один вектор `raw_features` — Spark ML принимает признаки только так |
| `StandardScaler` | делит каждый признак на его стандартное отклонение: коэффициенты становятся сравнимы |
| `LogisticRegression` | сама модель: вероятность класса 1 и прогноз 0/1 (порог 0,5) |

`pipeline.fit(train)` обучает все этапы по очереди и возвращает `PipelineModel`;
`model.transform(df)` добавляет колонки `probability` и `prediction`.

Здесь разбиение на train и test случайное: чеки одного дня друг от друга не зависят.

### Задача 2.2. Обучите логистическую регрессию

1. В `VectorAssembler` перечислите признаки: числовые `items`, `amount_rub`,
   `discount_pct`, `hour_msk` и векторы `payment_vec`, `format_vec`.
2. Создайте `LogisticRegression(labelCol="label", featuresCol="features")`.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression

train, test = receipts.randomSplit([0.8, 0.2], seed=42)

indexer = StringIndexer(inputCols=["payment_method", "format"], outputCols=["payment_idx", "format_idx"])
encoder = OneHotEncoder(inputCols=["payment_idx", "format_idx"], outputCols=["payment_vec", "format_vec"])

# Задача 2.2
assembler = VectorAssembler(inputCols=[...], outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features")
classifier = ...

pipeline = Pipeline(stages=[indexer, encoder, assembler, scaler, classifier])
model = pipeline.fit(train)
predictions = model.transform(test)
predictions.select("label", "probability", "prediction").show(5, truncate=False)

### Качество

* **Матрица ошибок** — сколько чеков каждого класса модель отнесла к каждому классу.
* **Accuracy** — доля верных ответов. Если бы участников было 95%, модель «всегда 1»
  дала бы 0,95 и ничему бы не научилась — поэтому смотрим и на другие метрики.
* **ROC AUC** — вероятность того, что случайный чек участника получит от модели более
  высокую вероятность, чем случайный чек не-участника. 0,5 — монетка, 1 — идеально.
  Не зависит от порога 0,5.

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

auc = BinaryClassificationEvaluator(labelCol="label").evaluate(predictions)
accuracy = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(predictions)
print(f"ROC AUC = {auc:.3f}   accuracy = {accuracy:.3f}   доля участников в test = "
      f"{test.filter('label = 1').count() / test.count():.3f}")

# строки — факт, колонки — прогноз модели
predictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

Коэффициенты модели. Признаки отмасштабированы, поэтому коэффициент — изменение
«уверенности» модели при росте признака на одно стандартное отклонение: чем больше по
модулю, тем сильнее признак. Плюс — за участника, минус — против. Для способа оплаты
и формата точка отсчёта — категория, выпавшая при one-hot (самая редкая).

In [ ]:
attrs = predictions.schema["raw_features"].metadata["ml_attr"]["attrs"]
names = [a["name"] for a in sorted((a for group in attrs.values() for a in group), key=lambda a: a["idx"])]
pd.Series(model.stages[-1].coefficients.toArray(), index=names).round(2).to_frame("коэффициент")

Вопросы:

1. Какой признак сильнее всего говорит «это участник программы»? Сходится ли это с
   таблицей средних выше?
2. Почему у `hour_msk` коэффициент почти 0?
3. Из матрицы ошибок посчитайте: какую долю участников модель нашла (**recall**) и какая
   доля чеков, отмеченных моделью, действительно принадлежит участникам (**precision**)?

*Дополнительно:* замените `LogisticRegression` на
`RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=50)` из
`pyspark.ml.classification`. Вырастет ли ROC AUC?

In [ ]:
receipts.unpersist()   # освобождаем память executor'ов

## Часть 3. Кластеризация магазинов: модель в Airflow

У сети 149 магазинов. Маркетингу нужны **сегменты** — группы похожих магазинов, чтобы
планировать ассортимент и акции для группы, а не для каждого магазина отдельно.
Правильного ответа заранее нет, целевой переменной тоже — это **обучение без учителя**,
алгоритм **k-means**.

Признаки магазина (одна строка = один магазин):

| признак | смысл |
|---|---|
| `receipts_per_day` | чеков в день — поток покупателей |
| `avg_check_rub` | средний чек, руб. |
| `items_per_receipt` | позиций в чеке — размер корзины |
| `cash_share_pct` | доля чеков, оплаченных наличными, % |

Модель переобучается **каждую неделю** DAG'ом
[dags/retail_ml_store_segments.py](../dags/retail_ml_store_segments.py), сам код — в
[jobs/retail/ml_store_segments.py](../jobs/retail/ml_store_segments.py). Откройте оба
файла. Ноутбук импортирует из скрипта функции `store_features` и `train`: число
сегментов подбирается на том же коде, что потом работает по расписанию.

In [ ]:
import ml_store_segments as seg

features = seg.store_features(spark).cache()
print(features.count(), "магазинов")
features.toPandas().head()

### Задача 3.1. Выберите число сегментов k

k-means сам не знает, сколько групп искать, — k задаёт аналитик. Две подсказки:

* **silhouette** (от −1 до 1): насколько магазин ближе к своему сегменту, чем к соседнему.
  Чем больше, тем чётче разделены сегменты;
* **интерпретируемость**: каждому сегменту можно дать понятное имя.

1. Для `k` от 2 до 8 обучите модель (`seg.train(features, k)` возвращает модель,
   магазины с колонкой `segment` и silhouette) и выведите silhouette и размеры сегментов
   (`segments.groupBy("segment").count()`).
2. Выберите `k`. Максимум silhouette — подсказка, а не ответ: сегмент из четырёх
   магазинов маркетингу вряд ли нужен. Сравните 4 и 7.
3. Для выбранного `k` посмотрите профили сегментов и их состав по форматам (ячейки ниже).

In [ ]:
# Задача 3.1
for k in range(2, 9):
    ...

In [ ]:
k = 4   # ваш выбор
model, segments, silhouette = seg.train(features, k)
stores = segments.toPandas()

print(f"k = {k}, silhouette = {silhouette:.3f}")
seg.profile(segments).toPandas().round(1)

In [ ]:
pd.crosstab(stores["segment"], stores["format"])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for s, group in stores.groupby("segment"):
    ax.scatter(group["avg_check_rub"], group["receipts_per_day"], label=f"сегмент {s}")
ax.set(xlabel="средний чек, руб.", ylabel="чеков в день", title="Магазины: средний чек и поток")
ax.legend()
plt.show()

Вопросы:

1. Дайте каждому сегменту короткое имя («гипермаркеты», «…»). Номера сегментов
   присвоены по возрастанию дневной выручки магазинов, поэтому от запуска к запуску
   не меняются — в отличие от номеров кластеров самого k-means.
2. Совпадают ли сегменты с форматами магазинов? Что, кроме формата, их разделяет?
   Подсказка: посмотрите `region` в `stores` для каждого сегмента.
3. На графике два сегмента накладываются друг на друга. Каким признаком они различаются?
4. Почему признаки обязательно масштабировать (`StandardScaler` в `train`)? Что было бы
   без этого с `receipts_per_day` (тысячи) и `cash_share_pct` (единицы)?

In [ ]:
features.unpersist()
spark.stop()   # освобождаем кластер для DAG

### Задача 3.2. Обучение по расписанию

DAG `retail_ml_store_segments` запускается **каждый понедельник в 06:00**
(`schedule="0 6 * * 1"`). Каждый запуск:

1. собирает признаки по свежим данным silver и обучает модель;
2. сохраняет модель в HDFS: `/models/store_segments/<дата>` — по версии на неделю;
3. перезаписывает таблицы `ml_store_segments` и `ml_segment_profile` в Postgres
   и **дописывает** строку в журнал `ml_runs` (дата, k, silhouette).

Шаги:

1. Если выбрали `k` не 4 — поменяйте `params={"k": 4}` в
   `dags/retail_ml_store_segments.py`. Запуски по расписанию берут это значение.
2. http://localhost:8080 → включите DAG. Из-за `catchup=False` Airflow сразу выполнит
   один запуск — за последнюю наступившую неделю, а не за все с `start_date`.
3. Попробуйте другое `k`, не меняя код: **Trigger DAG w/ config** → `{"k": 5}`.
4. Модели в HDFS: http://localhost:9870 → *Utilities → Browse the file system* →
   `/models/store_segments`.
5. Проверьте результат ячейкой ниже.

In [ ]:
spark = get_spark("lab-04-check")
read_from_dwh("ml_runs").orderBy("trained_at").show(truncate=False)
read_from_dwh("ml_segment_profile").orderBy("segment").show()

Сохранённую модель можно загрузить и применить к новым данным — например, к магазину,
который откроется в следующем месяце. Путь к последней модели берём из журнала `ml_runs`:

In [ ]:
from pyspark.ml import PipelineModel

model_path = read_from_dwh("ml_runs").orderBy(F.desc("trained_at")).first()["model_path"]
loaded = PipelineModel.load(model_path)

new_store = spark.createDataFrame(
    [(15000.0, 2200.0, 9.0, 12.0)],
    "receipts_per_day DOUBLE, avg_check_rub DOUBLE, items_per_receipt DOUBLE, cash_share_pct DOUBLE")
loaded.transform(new_store).select("cluster").show()   # номер кластера k-means, не сегмента
spark.stop()

## Часть 4. Дашборд «Сегменты магазинов» в Metabase

Откройте http://localhost:3000. Новые таблицы `ml_*` появятся после
⚙ **Admin settings → Databases → dwh → Sync database schema** (подождите полминуты и
обновите страницу).

**Вопрос 1 — состав сегментов.** **New → Question** → `Ml Store Segments` →
**Summarize:** `Count` по `Segment` и `Format` → **Visualization → Bar**, в настройках
включите **Stack**. Сохраните: *Сегменты и форматы магазинов*.

**Вопрос 2 — карта магазинов.** **New → SQL query** → база `dwh`:

```sql
SELECT avg_check_rub, receipts_per_day, 'Сегмент ' || segment AS segment, store_name
FROM ml_store_segments
```

**Visualization → Scatter**, в настройках ⚙ **Data**: X-axis `avg_check_rub`,
**Add series breakout** → `segment`, Y-axis `receipts_per_day`. Сохраните: *Средний
чек и поток покупателей*.

**Вопрос 3 — портрет сегментов.** `Ml Segment Profile` → **Visualization → Table**.
Сохраните: *Портрет сегментов*.

**Вопрос 4 — качество модели по неделям.** `Ml Runs` → **Summarize:** `Average of
Silhouette` по `Model Date: Day` → **Line**. Сохраните: *Silhouette по неделям*.

**Дашборд:** **New → Dashboard** → *Сегменты магазинов* → добавьте четыре вопроса.
Добавьте на дашборд текстовый блок с именами сегментов из задачи 3.1.

Когда DAG отработает в следующий понедельник, дашборд покажет новые сегменты сам, а на
графике silhouette появится новая точка.